# 연관 과목 이유 문장 엔진 실험 v3

v2 결론: LLM에게 "연결을 설명"하게 하면 근거에 없는 연결 주장을 27% 만든다. 프롬프트·재순위 검사로는 막지 못했다.

v3는 **역할을 나눈다.**
- 연결 부분은 **템플릿**: "이 연구실의 '{주제}' 연구와 이어져요. 이 과목에서 {내용}을 배워요."
- LLM은 **과목 근거 문장에서 가르치는 내용을 한국어 명사구로 옮기기만** 한다 (예: "PID Control" → "PID 제어").
- 검사는 **다국어 NLI**(문장 함의 판정)로 "이 수업은 {내용}을 다룬다"가 근거 문장에서 도출되는지 본다.
- 이유 생략은 재순위 점수가 아니라 **정보성 있는 근거 문장이 하나도 없을 때만** 한다.

| 방법 | 설명 |
|---|---|
| M2 | 영어 근거 인용 (기준) |
| M4b | v2 LLM 자유 생성 (기준) |
| M5b | M4b + 단어 비교 검사, 실패 시 M2 (v2 최고 안전형) |
| **M7** | 템플릿 + LLM 명사구 |
| **M8** | M7 + NLI 검사, 실패 시 M2 |

연구실·쌍은 v1·v2와 같다(12곳 × 상위 5개).

### 준비
- **런타임 → 런타임 유형 변경 → T4 GPU**
- 업로드 (서버 저장소 `data/`): `courses.jsonl`, `lab_profiles.jsonl`, `course_profiles.jsonl`, `concepts.json`
- 업로드 (서버 `experiments/bge_m3/V4/`): `rankings_v4.json`

### Kaggle 무인 실행 (gpuhub)
이 노트북은 Colab과 Kaggle 양쪽에서 돈다. VM에서 `gpuhub submit experiments/reason_engine/reason_engine_v3.ipynb` 로 제출하면 아래 입력 파일(저장소 루트 기준)이 비공개 데이터셋으로 올라가 자동 연결되고, 결과는 `gpuhub sync`로 받는다.


In [ ]:
# 1. GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('⚠ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 선택하고 다시 실행하세요.')

In [ ]:
# 2. 라이브러리 설치
!pip install -q sentence-transformers transformers accelerate bitsandbytes

In [ ]:
# 3. 데이터 준비 (Colab: 직접 업로드 / Kaggle: 연결된 데이터셋에서 복사)
import os, glob, shutil
need = ['courses.jsonl', 'lab_profiles.jsonl', 'course_profiles.jsonl', 'concepts.json', 'rankings_v4.json']
ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    for f in need:
        hits = glob.glob(f'/kaggle/input/**/{f}', recursive=True)
        if not os.path.exists(f) and hits:
            shutil.copy(hits[0], f)
missing = [f for f in need if not os.path.exists(f)]
if missing and not ON_KAGGLE:
    from google.colab import files
    print('업로드할 파일:', missing)
    files.upload()
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'아직 없는 파일: {missing}'
print('데이터 준비 완료', '(Kaggle)' if ON_KAGGLE else '(Colab)')

In [ ]:
# 4. 데이터와 평가 대상 쌍
import json, re, time, random, statistics
from collections import defaultdict, Counter
import numpy as np

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

courses = load_jsonl('courses.jsonl')
course_by_id = {c['record_id']: c for c in courses}
profiles = [p for p in load_jsonl('lab_profiles.jsonl') if p.get('status') != 'insufficient']
prof_by_id = {p['lab_id']: p for p in profiles}
course_profiles = {c['record_id']: c for c in load_jsonl('course_profiles.jsonl')}
concepts = {c['id']: c for c in json.load(open('concepts.json', encoding='utf-8'))['concepts']}
rankings = json.load(open('rankings_v4.json', encoding='utf-8'))['R2']

N_LABS, TOP_K, SEED = 12, 5, 7          # 연구실 수 × 상위 과목 수 = 평가 쌍 수
labs = random.Random(SEED).sample([p['lab_id'] for p in profiles], 20)[:N_LABS]   # v2~v4 채점 연구실과 같은 순서
pairs = [(lid, c['record_id']) for lid in labs for c in rankings[lid]['top30'][:TOP_K]]
print(f'평가 쌍 {len(pairs)}개 (연구실 {len(labs)}곳 × 상위 {TOP_K}개)')

def course_name(c):
    return c.get('course_name_kr') or c.get('course_name_en') or c['course_code']

## 1단계: 근거 선택 + 품질 기준

- 후보 문장에서 시험·과제·소개·개요·장 번호·"이 과목은/학생들은" 같은 일반 문장을 뺀다.
- 재순위 점수로 가장 잘 맞는 (주제 문장, 과목 문장)을 고른다.
- 정보성 있는 근거 문장이 하나도 없을 때만 **이유 생략**으로 표시한다(v2의 점수 백분위 기준은 필요한 과목까지 생략했다).

In [ ]:
from sentence_transformers import CrossEncoder
device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
if device == 'cuda':
    reranker.model.half()
def rr_scores(pairs_, bs=64):
    raw = np.asarray(reranker.predict(pairs_, batch_size=bs, show_progress_bar=len(pairs_) > 500), dtype=np.float32)
    return raw if (raw.min() >= 0 and raw.max() <= 1) else 1 / (1 + np.exp(-raw))

SENT_SPLIT = re.compile(r'(?<=[.!?。])\s+|(?<=다\.)\s*|\s*[•■▪\-–]\s+|\s*/\s+')
JUNK = re.compile(r'(see attached|attached file|첨부|tbd|tba|mid-?term|final|exam|quiz|homework|assignment|grading|attendance|presentation|review|holiday|no class|assumed|familiar with|prerequisite|중간|기말|시험|퀴즈|과제|출석|성적|평가|발표|휴강|공휴일|선수과목)', re.I)
GENERIC = re.compile(r'^(introduction|course (overview|introduction)|overview|class introduction.*|summary|conclusion|basics?|chapter.*|\(?chapter.*|ch\.?\s*\d.*|part [ivx\d]+.*|the (course|objective|goal) (material|of this (class|course)).*|(this|the) course (will|is|aims|covers|provides|introduces|deals).*|this class .*|in this (course|class).*|students will.*|it is hoped.*|본 (과목|강의|교과목)(은|에서는|을).*|이 (과목|강의)(은|에서는).*|학생들은.*)$', re.I)
CONTENT = re.compile(r'[A-Z]{2,}|[A-Za-z]{4,}|[가-힣]{2,}')
STOPW = {'this', 'that', 'with', 'from', 'will', 'course', 'class', 'students', 'student', 'learn', 'introduction', 'overview', 'basic', 'basics',
         '과목', '강의', '학습', '이해', '소개', '기본', '기초', '내용', '학생'}

def informative(s):
    if JUNK.search(s) or GENERIC.match(s.strip()):
        return False
    words = [w for w in CONTENT.findall(s) if w.lower() not in STOPW]
    return len(words) >= 2 or (len(words) == 1 and len(words[0]) >= 8)

def course_sentences(c):
    out = []
    for where, text in [('과목 설명', c.get('description') or ''), ('학습목표', c.get('objectives') or '')]:
        for s in SENT_SPLIT.split(text):
            s = re.sub(r'\s+', ' ', s).strip()
            if 15 <= len(s) <= 300 and informative(s):
                out.append((where, s))
    for line in re.split(r'\n+', str(c.get('course_plan') or '')):
        line = re.sub(r'^[\W\d_]*(week|주차|lecture|chapter)?\s*[\d.]*\s*[:.)-]?\s*', '', line.strip(), flags=re.I)
        line = re.sub(r'\s+', ' ', line)
        if 12 <= len(line) <= 200 and informative(line):
            out.append(('강의계획', line))
    return list(dict.fromkeys(out))[:60]

def primary_topics(lid):
    return [t for t in prof_by_id[lid]['topics'] if t.get('weight', 'primary') == 'primary']

t0 = time.time()
batch, owners = [], []
for pi, (lid, rid) in enumerate(pairs):
    sents = course_sentences(course_by_id[rid])
    for ti, t in enumerate(primary_topics(lid)):
        for si, (where, s) in enumerate(sents):
            batch.append((f"{t['label_ko']}: {t['quote']}", s))
            owners.append((pi, ti, si))
scores = rr_scores(batch)
best = {}
for (pi, ti, si), sc in zip(owners, scores):
    if pi not in best or sc > best[pi][0]:
        best[pi] = (float(sc), ti, si)

evidence = []
for pi, (lid, rid) in enumerate(pairs):
    c = course_by_id[rid]; sents = course_sentences(c)
    e = {'lab_id': lid, 'record_id': rid, 'lab': prof_by_id[lid]['professors'][0].split(' (')[0],
         'lab_summary': prof_by_id[lid]['summary_ko'], 'course': course_name(c), 'course_en': c.get('course_name_en') or '',
         'code': c['course_code']}
    if pi in best:
        sc, ti, si = best[pi]; t = primary_topics(lid)[ti]; where, s = sents[si]
        e.update({'topic': t['label_ko'], 'topic_quote': t['quote'], 'where': where, 'sentence': s, 'rr': round(sc, 4), 'skip': False})
    else:
        t = primary_topics(lid)[0]
        e.update({'topic': t['label_ko'], 'topic_quote': t['quote'], 'where': '', 'sentence': '', 'rr': round(best.get(pi, (0.0,))[0], 4), 'skip': True})
    evidence.append(e)
print(f'근거 선택 {len(batch)}쌍 {time.time()-t0:.1f}s')
print(f"이유 생략 {sum(e['skip'] for e in evidence)} / {len(evidence)}쌍")
for e in evidence[:5]:
    print(f"  [{e['lab']}] {e['topic']} ↔ {e['course']}: {'(생략)' if e['skip'] else e['where'] + ' — ' + e['sentence'][:80]}")

## 2단계: LLM 로드, 명사구 추출(M7용), 자유 생성(M4b 기준)

기본 모델은 `Qwen/Qwen2.5-7B-Instruct`(4비트). `LLM_ID`만 바꿔 다른 모델과 비교할 수 있다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLM_ID = 'Qwen/Qwen2.5-7B-Instruct'
import gc
for _name in ['llm', 'tok']:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU 여유 메모리 {free/1e9:.1f} / {total/1e9:.1f} GB (7B 4비트에 약 6GB 필요)')
    if free < 6.5e9:
        print('⚠ 메모리가 부족합니다. 런타임 → 세션 다시 시작 후 처음부터 실행하세요.')
t0 = time.time()
tok = AutoTokenizer.from_pretrained(LLM_ID, trust_remote_code=True)
tok.padding_side = 'left'
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID, device_map={'': 0} if torch.cuda.is_available() else 'cpu', trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'))
print(f'LLM 로드 {time.time()-t0:.1f}s')

def generate(messages_list, max_new=120, bs=8):
    outs = []
    for i in range(0, len(messages_list), bs):
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list[i:i+bs]]
        enc_in = tok(texts, return_tensors='pt', padding=True).to(llm.device)
        with torch.no_grad():
            out = llm.generate(**enc_in, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        for ids in out[:, enc_in['input_ids'].shape[1]:]:
            outs.append(tok.decode(ids, skip_special_tokens=True).strip())
        print(f'  {min(i+bs, len(messages_list))}/{len(messages_list)}')
    return outs

HAN = re.compile(r'[\u4e00-\u9fff\u3040-\u30ff]')
def clean(text):
    t = text.strip().split(chr(10))[0].strip().strip('"').strip()
    t = HAN.sub('', t)
    t = re.sub(r'_[A-Za-z]+', '', t)
    t = re.sub(r'\s{2,}', ' ', t).replace('。', '.').strip()
    return t

active = [e for e in evidence if not e['skip']]

# --- M7용 명사구 추출: 연결은 말하지 않고 '무엇을 가르치는가'만 옮긴다 ---
PH_SYS = ('강의 자료 문장에서 이 수업이 가르치는 핵심 내용을 한국어 명사구 하나(25자 이내)로 옮겨라. '
          '문장에 없는 내용을 더하지 마라. 전공 용어는 널리 쓰는 한국어 용어를 쓰고, 필요하면 괄호에 영어 원어를 붙여라. '
          '명사구만 출력하라. 마침표·따옴표·조사를 붙이지 마라.')
PH_SHOTS = [('PID Control', 'PID 제어'), ('Protein Composition and Structure', '단백질의 구성과 구조'),
            ('Learning with multi-modal data (text and audio)', '텍스트·오디오 멀티모달 데이터 학습'),
            ('CV: Deep learning for visual recognition', '시각 인식을 위한 딥러닝'),
            ('데이터 베이스의각 모델, 기술방법, 실현방법에 대하여 배운다.', '데이터베이스 모델과 구현 방법')]
def ph_messages(e):
    msgs = [{'role': 'system', 'content': PH_SYS}]
    for src, dst in PH_SHOTS:
        msgs += [{'role': 'user', 'content': src}, {'role': 'assistant', 'content': dst}]
    return msgs + [{'role': 'user', 'content': e['sentence']}]
t0 = time.time()
phs = generate([ph_messages(e) for e in active], max_new=40)
for e, ph in zip(active, phs):
    p = clean(ph).strip(" .,'\"·")
    p = re.sub(r'(을|를|이|가|은|는|에 대한|에 대해)$', '', p).strip()
    e['phrase'] = p
    e['sentence_ko'] = e['sentence'] + ' / ' + p      # M4b·단어 비교 검사가 번역어도 근거로 보도록
print(f'명사구 {len(active)}건 {time.time()-t0:.1f}s')
for e in active[:6]:
    print(f"   {short(e['sentence'], 50) if 'short' in globals() else e['sentence'][:50]}  →  {e['phrase']}")

# --- M4b: v2와 같은 자유 생성 (비교 기준) ---
SYSTEM = ('너는 대학생에게 연구실과 수업의 연결을 설명하는 도우미다. [근거]에 적힌 내용만 사용해 한국어 해요체 한 문장(40~80자)으로 '
          '이 수업에서 배우는 것이 이 연구실 주제와 어떻게 이어지는지 설명하라.\n'
          '규칙: 근거에 없는 사실·용어·연구실 프로젝트·실험을 만들지 마라. "직접", "반드시", "필수", "최고" 같은 단정 표현을 쓰지 마라. '
          '영어 단어를 한국어 문장 중간에 섞지 말고 한국어 용어를 써라. 문장만 출력하라.')
def m4_messages(e):
    user = (f"[근거]\n연구실 주제: {e['topic']}\n연구실 자료 문장: {e['topic_quote']}\n"
            f"수업: {e['course']}\n수업 자료({e['where']}): {e['sentence']} ({e['phrase']})")
    return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user}]
t0 = time.time()
gens = generate([m4_messages(e) for e in active])
for e, g_ in zip(active, gens):
    e['M4b'] = clean(g_)
print(f'M4b 생성 {time.time()-t0:.1f}s')

## 3단계: 템플릿 조립(M7)과 NLI 검사(M8)

- **M7**: "이 연구실의 '{주제}' 연구와 이어져요. 이 과목에서 {명사구}을/를 배워요." 조사는 받침에 맞춰 붙인다.
- **NLI 검사**: 전제 = 근거 원문, 가설 = "이 수업은 {명사구}을/를 다룬다." 함의 확률이 `NLI_MIN` 이상이면 통과.
  모델: `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`(다국어, MIT 라이선스).
- **M8**: NLI 통과 시 M7, 실패하거나 명사구가 이상하면(비었거나 30자 초과, 깨짐) M2.
- **M5b**: v2와 같은 단어 비교 검사(M4b → 실패 시 M2). 비교 기준.

In [ ]:
def short(s, n=70):
    s = re.sub(r'\s+', ' ', s).strip()
    return s if len(s) <= n else s[:n].rsplit(' ', 1)[0] + '…'
def josa(word, pair=('을', '를')):
    base = re.sub(r'\s*\([^)]*\)\s*$', '', word or '').rstrip()   # 괄호 병기는 앞 한글 기준
    ch = base[-1:]
    if ch and '가' <= ch <= '힣':
        return pair[0] if (ord(ch) - 0xAC00) % 28 else pair[1]
    return f'{pair[0]}({pair[1]})'

FRAME = {'연구', '연구실', '과목', '수업', '배우', '배워', '배울', '배우면', '다뤄', '다루', '연결', '도움', '필요', '관련', '지식', '내용',
         '이해', '학습', '활용', '기초', '기본', '바탕', '주제', '분야', '통해', '대한', '위한', '있어요', '있습니다', '있는', '하는', '돼요', '이에요', '에서', '개념', '이어져요', '이어지'}
OVERCLAIM = re.compile(r'(반드시|필수|최고|완벽|보장|무조건|직접적으로|직접 적용|핵심 도구|중요한 도구)')
GARBLE_RE = re.compile(r'([\u4e00-\u9fff]|_[A-Za-z]|[A-Za-z]{3,}[가-힣]|[가-힣][A-Za-z]{3,}|\b[a-z]{3,}\b)')
class _Garble:
    def search(self, t):
        return GARBLE_RE.search(re.sub(r'\([^)]*\)', '', t))
GARBLE = _Garble()
def content_words(s):
    return [w for w in re.findall(r'[A-Za-z]{3,}|[가-힣]{2,}', s) if w not in FRAME and not any(w.startswith(f) for f in FRAME)]
def unsupported_ratio(text, e):
    src = ' '.join([e['topic'], e['topic_quote'], e['course'], e['course_en'], e['sentence'], e['sentence_ko']]).lower()
    words = content_words(text)
    if not words:
        return 0.0
    return sum(w.lower() not in src and w[:2].lower() not in src for w in words) / len(words)

for e in active:
    e['M2'] = f"이 연구실의 '{e['topic']}' 연구와 연결돼요. 이 과목의 {e['where']}에서 \"{short(e['sentence'])}\"를 다뤄요."
    e['M7'] = f"이 연구실의 '{e['topic']}' 연구와 이어져요. 이 과목에서 {e['phrase']}{josa(e['phrase'])} 배워요."
    hard_bad = bool(OVERCLAIM.search(e['M4b']) or GARBLE.search(e['M4b']) or len(e['M4b']) < 15)
    e['M5b_fallback'] = bool(hard_bad or unsupported_ratio(e['M4b'], e) > 0.35)
    e['M5b'] = e['M2'] if e['M5b_fallback'] else e['M4b']

from transformers import AutoModelForSequenceClassification
NLI_ID = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
nli_tok = AutoTokenizer.from_pretrained(NLI_ID)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_ID).to(device).eval()
ent_idx = [i for i, l in nli.config.id2label.items() if 'entail' in l.lower()][0]
def entail_prob(premises, hyps, bs=32):
    out = []
    for i in range(0, len(premises), bs):
        enc_in = nli_tok(premises[i:i+bs], hyps[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            probs = torch.softmax(nli(**enc_in).logits.float(), dim=-1)[:, ent_idx]
        out += probs.cpu().tolist()
    return np.array(out)
hyps = [f"이 수업은 {e['phrase']}{josa(e['phrase'])} 다룬다." for e in active]
ent = entail_prob([e['sentence'] for e in active], hyps)
NLI_MIN = 0.5
for e, p in zip(active, ent):
    bad_phrase = (not e['phrase']) or len(e['phrase']) > 30 or bool(GARBLE.search(e['phrase'])) or bool(re.search(r'(요|다|니다)\.?$', e['phrase']))
    e['nli'] = round(float(p), 4)
    e['M8_fallback'] = bool(bad_phrase or p < NLI_MIN)
    e['M8'] = e['M2'] if e['M8_fallback'] else e['M7']
print(f'NLI 함의 확률 분포(10/50/90%): {[round(float(np.percentile(ent, q)), 3) for q in (10, 50, 90)]}')
print(f"대체율: M5b {np.mean([e['M5b_fallback'] for e in active]):.0%}, M8 {np.mean([e['M8_fallback'] for e in active]):.0%}")

METHODS = ['M2', 'M4b', 'M5b', 'M7', 'M8']
import pandas as pd
auto = {}
for m in METHODS:
    texts = [e[m] for e in active]
    auto[m] = {'평균 길이(자)': round(statistics.mean(len(t) for t in texts), 1),
               '근거 밖 단어 비율': round(statistics.mean(unsupported_ratio(t, e) for t, e in zip(texts, active)), 3),
               '과장 표현(건)': sum(bool(OVERCLAIM.search(t)) for t in texts),
               '깨진 글자(건, M2는 영어 인용이라 제외)': None if m == 'M2' else sum(bool(GARBLE.search(t)) for t in texts),
               '서로 다른 문장 비율': round(len(set(texts)) / len(texts), 3)}
print(f"이유 생략 쌍: {sum(e['skip'] for e in evidence)} (평가 대상 {len(active)}쌍)")
pd.DataFrame(auto)

In [ ]:
# 나란히 보기
for e in active[:8]:
    print(f"\n■ {e['lab']} · {e['topic']} ↔ {e['code']} {e['course']}  (근거 {e['rr']}, NLI {e['nli']})")
    print(f"   근거: {e['where']} — {short(e['sentence'], 80)}")
    print(f"   명사구: {e['phrase']}")
    for m in METHODS:
        print(f'   {m}: {e[m]}')
print('\n이유 생략된 쌍 예시:')
for e in [x for x in evidence if x['skip']][:5]:
    print(f"   {e['lab']} ↔ {e['code']} {e['course']} (정보성 있는 근거 문장 없음)")

## 4단계: Claude Code 채점용 파일 (블라인드)

v1과 같은 기준: **faithful** 0/1, **specific**·**fluent**·**useful** 1~3. 이유 생략된 쌍은 채점에서 빠지고, 생략이 적절했는지 따로 0/1로 채점한다(`skip_ok`).

In [ ]:
packet, key = [], {}
for pi, e in enumerate(evidence):
    base = {'pair': f'p{pi:03d}', 'lab': e['lab'], 'lab_summary': e['lab_summary'], 'topic': e['topic'],
            'topic_quote': e['topic_quote'][:250], 'course': f"{e['code']} {e['course']} / {e['course_en']}",
            'course_description': (course_by_id[e['record_id']].get('description') or '')[:300]}
    if e['skip']:
        packet.append({**base, 'skipped': True, 'reasons': [{'id': f'p{pi:03d}|skip', 'text': '(근거가 약해 이유 문장을 생략함)'}]})
        key[f'p{pi:03d}|skip'] = ['SKIP']
        continue
    texts = {}
    for m in METHODS:
        texts.setdefault(e[m], []).append(m)
    items = list(texts.items())
    random.Random(pi).shuffle(items)
    reasons = []
    for j, (t, ms) in enumerate(items):
        rid = f'p{pi:03d}|r{j+1}'
        reasons.append({'id': rid, 'text': t}); key[rid] = ms
    packet.append({**base, 'skipped': False, 'evidence_sentence': e['sentence'], 'reasons': reasons})
json.dump({'instructions': '각 reason에 faithful(0/1), specific(1~3), fluent(1~3), useful(1~3). '
                           'skipped 쌍의 "|skip" 항목에는 skip_ok(0/1: 생략이 적절하면 1). '
                           '반환 형식: {"<id>": {"faithful":..,"specific":..,"fluent":..,"useful":..}} 또는 {"<id>|skip": {"skip_ok":0|1}}',
           'pairs': packet}, open('reason_packet.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump(key, open('reason_key.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(f'채점용 파일: 쌍 {len(packet)}개(생략 {sum(p["skipped"] for p in packet)}), 항목 {len(key)}개 → reason_packet.json')

## 결과 저장·다운로드

In [ ]:
import zipfile
json.dump(evidence, open('reasons_v3.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
pd.DataFrame(auto).to_csv('reason_auto_metrics_v3.csv', encoding='utf-8-sig')
json.dump({'N_LABS': N_LABS, 'TOP_K': TOP_K, 'SEED': SEED, 'LLM_ID': LLM_ID, 'NLI_ID': NLI_ID, 'NLI_MIN': NLI_MIN, 'RERANKER': 'BAAI/bge-reranker-v2-m3',
           'PH_SHOTS': PH_SHOTS},
          open('reason_settings_v3.json', 'w'), indent=1)
with zipfile.ZipFile('reason_results_v3.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['reasons_v3.json', 'reason_auto_metrics_v3.csv', 'reason_settings_v3.json', 'reason_packet.json', 'reason_key.json']:
        z.write(f)
print('저장 완료: reason_results_v3.zip')
if not ON_KAGGLE:
    files.download('reason_results_v3.zip')   # Kaggle 에서는 /kaggle/working 의 파일이 그대로 결과가 된다